In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,DateType,TimestampType, FloatType

from pyspark.sql.functions import to_timestamp, date_format, col

from pyspark.sql.functions import *

catalog_name = "openaq"

schema = "silver"

In [0]:
df_locations=spark.read.table(f"{catalog_name}.{schema}.slv_locations")
df_licenses=spark.read.table(f"{catalog_name}.{schema}.slv_licenses")
df_instruments=spark.read.table(f"{catalog_name}.{schema}.slv_instruments")

### Merging Location and Licenses

In [0]:
df_locations.columns

In [0]:
df_licenses.columns

In [0]:
df_location_license  = spark.sql(f""" 
SELECT 
    loc.location_id, loc.std_location_name, loc.std_locality, loc.latitude, loc.longitude, loc.country_id, loc.std_country_name, loc.std_country_code, loc.owner_name, loc.provider_id, loc.isMobile, loc.isMonitor, loc.instrument_id, loc.std_owner_name, loc.std_provider_name, loc.sensor_id, loc.sensor_name, loc.parameter_id, loc.parameter_name, loc.parameter_units, loc.parameter_displayName, loc.license_id,  loc.std_license_name, loc.std_license_attribution_name, loc.std_instrument_name, loc.std_timezone, loc.std_first_seen_utc, loc.std_last_seen_utc,
    lic.attributionRequired, lic.commercialUseAllowed, lic.modificationAllowed, lic.redistributionAllowed, lic.shareAlikeRequired
FROM {catalog_name}.{schema}.slv_locations AS loc
LEFT JOIN {catalog_name}.{schema}.slv_licenses AS lic
    ON loc.license_id = lic.license_id
   AND loc.std_license_name = lic.std_license_name
""")

### Merging Locations, Licenses and Instruments

In [0]:
display(df_location_license)

In [0]:
df_instruments.columns

In [0]:
df_location_license.createOrReplaceTempView("df_location_license")
gold_dim_consolidated  = spark.sql(f"""
SELECT 
    ll.*,
    i.manufacturer_id,
    i.std_manufacturer_name
FROM df_location_license AS ll
LEFT JOIN {catalog_name}.{schema}.slv_instruments AS i
    ON ll.instrument_id = i.instrument_id
""")

In [0]:
display(gold_dim_consolidated)

In [0]:
gold_dim_consolidated.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog_name}.gold.gld_dim_consolidated")

In [0]:
gold_dim_consolidated.columns